# 手写全连接神经网络

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim # 梯度下降（优化器）
from torchvision import datasets,transforms
import numpy as np

In [3]:
batch_size = 64
learning_rate = 0.01
num_epochs = 30
transform = transforms.Compose([
    transforms.ToTensor(),
])
train_dataset = datasets.MNIST(root="./data",train=True,download=True,transform=transform) # 60K = 6w
test_dataset = datasets.MNIST(root="./data",train=False,download=True,transform=transform) # 10K = 1w
train_loader = torch.utils.data.DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset,batch_size=batch_size,shuffle=False)

In [4]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN,self).__init__()
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(in_features=28*28,out_features=128)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(in_features=128,out_features=10)
    def forward(self,x):
        out = self.flatten(x)
        out = self.linear1(out)
        out = self.relu(out)
        out = self.linear2(out)
        return out
model = SimpleNN()
model.train()

SimpleNN(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear1): Linear(in_features=784, out_features=128, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=128, out_features=10, bias=True)
)

In [5]:
loss = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(),lr=learning_rate)

In [6]:
for epoch in range(num_epochs):
    loss_num = 0.0
    for i,(image,lable) in enumerate(train_loader):
        out = model(image)
        l = loss(out,lable)
        optimizer.zero_grad()
        l.backward()
        optimizer.step()
        loss_num += l.item()
        if (i + 1) % 100 == 0:
            print(f"第{epoch+1}轮,第{i+1}次,loss:{loss_num/100}")
            loss_num = 0.0

第1轮,第100次,loss:2.2102646589279176
第1轮,第200次,loss:1.9649226582050323
第1轮,第300次,loss:1.6387319207191466
第1轮,第400次,loss:1.330096869468689
第1轮,第500次,loss:1.073378106355667
第1轮,第600次,loss:0.9130239760875702
第1轮,第700次,loss:0.7830965882539749
第1轮,第800次,loss:0.7044599306583404
第1轮,第900次,loss:0.6449593120813369
第2轮,第100次,loss:0.5608196040987968
第2轮,第200次,loss:0.5417391556501389
第2轮,第300次,loss:0.523406785428524
第2轮,第400次,loss:0.5141765883564949
第2轮,第500次,loss:0.4904778625071049
第2轮,第600次,loss:0.4632039031386375
第2轮,第700次,loss:0.4668737190961838
第2轮,第800次,loss:0.4579944133758545
第2轮,第900次,loss:0.43933172434568407
第3轮,第100次,loss:0.4170031617581844
第3轮,第200次,loss:0.4010914251208305
第3轮,第300次,loss:0.41795424729585645
第3轮,第400次,loss:0.39853581085801126
第3轮,第500次,loss:0.3958778116106987
第3轮,第600次,loss:0.3887755560874939
第3轮,第700次,loss:0.38466719090938567
第3轮,第800次,loss:0.38207250863313674
第3轮,第900次,loss:0.3707405376434326
第4轮,第100次,loss:0.35960471734404564
第4轮,第200次,loss:0.3468874632567167
第4轮,第300次,l

In [9]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs,lables in test_loader:
        outputs = model(inputs)
        predicted = torch.argmax(outputs,dim=1)
        total += lables.size(0)
        correct += (predicted == lables).sum().item()
acc = (correct / total) * 100
print(f"Acc of the network on test dataset is {acc}")

Acc of the network on test dataset is 95.49


# 手写CNN

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim # 梯度下降（优化器）
from torchvision import datasets,transforms
import numpy as np

In [11]:
batch_size = 64
learning_rate = 0.01
num_epochs = 30
transform = transforms.Compose([
    transforms.ToTensor(),
])
train_dataset = datasets.MNIST(root="./data",train=True,download=True,transform=transform) # 60K = 6w
test_dataset = datasets.MNIST(root="./data",train=False,download=True,transform=transform) # 10K = 1w
train_loader = torch.utils.data.DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset,batch_size=batch_size,shuffle=False)

In [12]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1,out_channels=16,kernel_size=3,stride=1,padding=1)
        self.relu1 = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(in_channels=16,out_channels=32,kernel_size=3,stride=1,padding=1)
        self.relu2 = nn.ReLU()
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(in_features=32*7*7,out_features=128)
        self.relu3 = nn.ReLU()
        self.linear2 = nn.Linear(in_features=128,out_features=10)
    def forward(self,x):
        out = self.conv1(x)
        out = self.relu1(out)
        out = self.pool(out)
        out = self.conv2(out)
        out = self.relu2(out)
        out = self.pool(out)
        out = self.flatten(out)
        out = self.linear1(out)
        out = self.relu3(out)
        out = self.linear2(out)
        return out
model = CNN()
model.train()

CNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu1): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu2): ReLU()
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear1): Linear(in_features=1568, out_features=128, bias=True)
  (relu3): ReLU()
  (linear2): Linear(in_features=128, out_features=10, bias=True)
)

In [13]:
loss = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(),lr=learning_rate)

In [14]:
for epoch in range(num_epochs):
    loss_num = 0.0
    for i,(image,lable) in enumerate(train_loader):
        out = model(image)
        l = loss(out,lable)
        optimizer.zero_grad()
        l.backward()
        optimizer.step()
        loss_num += l.item()
        if (i + 1) % 100 == 0:
            print(f"第{epoch+1}轮,第{i+1}次,loss:{loss_num/100}")
            loss_num = 0.0

第1轮,第100次,loss:2.294649558067322
第1轮,第200次,loss:2.259691183567047
第1轮,第300次,loss:2.1552218222618102
第1轮,第400次,loss:1.7018606817722322
第1轮,第500次,loss:0.914735016822815
第1轮,第600次,loss:0.6336227589845658
第1轮,第700次,loss:0.5175042518973351
第1轮,第800次,loss:0.44428238183259966
第1轮,第900次,loss:0.41351227179169653
第2轮,第100次,loss:0.3779754942655563
第2轮,第200次,loss:0.3617541944980621
第2轮,第300次,loss:0.3322607818245888
第2轮,第400次,loss:0.3455953448265791
第2轮,第500次,loss:0.3044203962385654
第2轮,第600次,loss:0.29455513015389445
第2轮,第700次,loss:0.2801252331584692
第2轮,第800次,loss:0.27567025676369666
第2轮,第900次,loss:0.27186205610632896
第3轮,第100次,loss:0.26417021691799164
第3轮,第200次,loss:0.24674686670303345
第3轮,第300次,loss:0.2454681397974491
第3轮,第400次,loss:0.21830745041370392
第3轮,第500次,loss:0.21047515455633403
第3轮,第600次,loss:0.20674751177430153
第3轮,第700次,loss:0.2058993224427104
第3轮,第800次,loss:0.1948587343841791
第3轮,第900次,loss:0.205268190279603
第4轮,第100次,loss:0.19440089035779237
第4轮,第200次,loss:0.17622851949185134
第4轮,第3

In [19]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images,lables in test_loader:
        outputs = model(images)
        predicted = torch.argmax(outputs,dim=1)
        total += lables.size(0)
        correct += (predicted == lables).sum().item()
print(f"Acc of the network on test dataset is {(correct/total)*100}")

Acc of the network on test dataset is 98.54
